## Program 1B

The same implementation of theory in program 1A, but the matrices have been "chunked" to avoid overwhelming the RAM. Lap timer points reduced, as steps become merged under segmentation.

In [3]:
n_rows = 1

In [4]:
import pandas as pd
import numpy as np
import time
from time import perf_counter_ns

h_1 = 20
r_1 = 20
h_2 = 10
r_2 = 20

file_path = r"C:\Users\Smith\OneDrive\MSc Project\05 Final Code\Bunny Head Raw Lines for Computation Testing.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name=0,
    usecols="A:F",
    nrows=n_rows,
    header=None,
    engine="openpyxl"
)

df.columns = ["x", "y", "z", "i", "j", "k"]
A = df.to_numpy(dtype=float)

In [5]:
def point_in_CTC_chunked(A, h_1, r_1, h_2, r_2, eps=1e-12, row_block=256, col_block=2048):
    
    t_start = perf_counter_ns()
    t_last = t_start

    def lap(name):
        nonlocal t_last
        now = perf_counter_ns()
        dt_ms = (now - t_last) / 1_000_000
        total_ms = (now - t_start) / 1_000_000
        print(f"{name:<45} /{dt_ms:10.3f}/ ms   total: {total_ms:10.3f} ms")
        t_last = now

    A = np.asarray(A, dtype=np.float64)
    lap("Input to numpy array")

    P = A[:, 0:3]   # xyz points
    U = A[:, 3:6]   # unit orientation vectors
    N = A.shape[0]
    lap("Separate Points and Vectors")

    # Dot product calculations, converted from the N x N matrix process
    p2 = np.einsum("ij,ij->i", P, P)
    tip_dp_axis = np.einsum("ij,ij->i", P, U)
    lap("Calculate p2 and tip dot axis")

    hit_chunks = []

    h_total = h_1 + h_2

    h1_sq = h_1 * h_1
    r1_sq = r_1 * r_1
    r2_sq = r_2 * r_2

    loop_start = perf_counter_ns()

    for i0 in range(0, N, row_block):
        i1 = min(i0 + row_block, N)

        P_i = P[i0:i1]
        U_i = U[i0:i1]
        p2_i = p2[i0:i1]
        tip_i = tip_dp_axis[i0:i1]

        row_idx = np.arange(i0, i1)

        # Only need to check against columns up to i1.
        for j0 in range(0, i1, col_block):
            j1 = min(j0 + col_block, i1)

            P_j = P[j0:j1]
            p2_j = p2[j0:j1]
            col_idx = np.arange(j0, j1)

            # t = dot(P_j, U_i) - dot(P_i, U_i), changed from 1A
            t = U_i @ P_j.T
            t -= tip_i[:, None]

            # Squared distance between tip point P_i and candidate point P_j
            dot_PP = P_i @ P_j.T
            v2 = p2_i[:, None] + p2_j[None, :] - 2.0 * dot_PP
            np.maximum(v2, 0.0, out=v2)

            # d_perp_sq = v2 - t^2
            d_perp_sq = np.empty_like(v2)
            np.multiply(t, t, out=d_perp_sq)
            np.subtract(v2, d_perp_sq, out=d_perp_sq)
            np.maximum(d_perp_sq, 0.0, out=d_perp_sq)

            # Axial slab test
            axial_ok = (t >= -eps) & (t <= h_total + eps)

            # Cone radial test
            t_sq = np.empty_like(t)
            np.multiply(t, t, out=t_sq)

            cone_ok = (h1_sq * d_perp_sq) <= (r1_sq * t_sq + eps)

            # Cylinder radial test
            cylinder_ok = d_perp_sq <= (r2_sq + eps)

            cone_region = t <= h_1 + eps
            cylinder_region = t > h_1 + eps

            radial_ok = (cone_region & cone_ok) | (cylinder_region & cylinder_ok)

            tile_inside = axial_ok & radial_ok

            # Mask for where j < i
            if j1 <= i0:
                pass
            else:
                previous_mask = col_idx[None, :] < row_idx[:, None]
                tile_inside &= previous_mask

            local_hits = np.argwhere(tile_inside)

            if local_hits.size:
                local_hits[:, 0] += i0
                local_hits[:, 1] += j0
                hit_chunks.append(local_hits)

    loop_ms = (perf_counter_ns() - loop_start) / 1_000_000
    lap("Chunked matrix processing")

    if hit_chunks:
        hit_pairs = np.vstack(hit_chunks)
    else:
        hit_pairs = np.empty((0, 2), dtype=np.int64)

    lap("Hit list")

    print("-" * 75)
    print(f"{'TOTAL':<45} {(perf_counter_ns() - t_start) / 1_000_000:10.3f} ms")

    return hit_pairs

In [6]:
start = time.perf_counter()
hit_pairs = point_in_CTC_chunked(A, h_1, r_1, h_2, r_2, row_block=256, col_block=2048)
elapsed_ms = (time.perf_counter() - start) * 1000
print(f"Function time: {elapsed_ms:.3f} ms")
print(hit_pairs.shape)

Input to numpy array                          /     0.023/ ms   total:      0.023 ms
Separate Points and Vectors                   /     0.950/ ms   total:      0.973 ms
Calculate p2 and tip dot axis                 /     1.043/ ms   total:      2.016 ms
Chunked matrix processing                     /    15.438/ ms   total:     17.453 ms
Hit list                                      /     0.110/ ms   total:     17.563 ms
---------------------------------------------------------------------------
TOTAL                                             17.642 ms
Function time: 24.045 ms
(0, 2)
